In [ ]:
import pylustrator
pylustrator.start()
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm

# Load the saved clusters from the specified iterations
initial_clusters = np.load('data_array_beta.npy')
labels = initial_clusters[:, 11]

xyz_initial = initial_clusters[:, :3]
labels_initial = initial_clusters[:, 11]

# Set plot limits and grid size
x_limits = (158, 190)
y_limits = (112, 160)
pixel_size_mm = 2

# Create a 2x2 subplot for double-column format
fig, axs = plt.subplots(1, 2, figsize=(12, 8), dpi=300, constrained_layout=True)

unique_labels = np.unique(labels_initial)
label_to_color = {label: idx for idx, label in enumerate(unique_labels)}

# Function to add rectangle patches for each point
def add_rectangles(ax, xyz_data, labels, cmap):
    for (x, y, _), label in zip(xyz_data, labels):
        color = cmap(label_to_color[label])
        rect = patches.Rectangle((x, y), pixel_size_mm, pixel_size_mm, linewidth=0.5,
                                 edgecolor='none', facecolor=color, alpha=0.7)
        ax.add_patch(rect)

# Function to set custom grid and sparse Y-axis tick labels
def set_custom_grid(ax):
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
    
    # Set grid ticks every 2 mm on both axes
    ax.set_xticks(np.arange(x_limits[0], x_limits[1] + 1, pixel_size_mm), minor=False)
    ax.set_yticks(np.arange(y_limits[0], y_limits[1] + 1, pixel_size_mm), minor=True)
    ax.set_xticks(np.arange(x_limits[0], x_limits[1] + 1, pixel_size_mm), minor=True)
    ax.set_yticks(np.arange(y_limits[0], y_limits[1] + 1, 8), minor=False)
    ax.set_xticks(np.arange(x_limits[0], x_limits[1] + 1, 8), minor=False)
    
    # Display grid
    ax.grid(which="both", color="gray", linestyle="--", linewidth=0.5)

    # Label Y-axis major ticks every 8 mm
    y_labels = np.arange(y_limits[0], y_limits[1] + 1, 8)
    ax.set_yticklabels(y_labels)

    # Add thicker grid lines between Y = 122 and Y = 132
    ax.axhline(y=122, color='black', linewidth=2, linestyle='--')  # Thicker grid line at Y=122
    ax.axhline(y=132, color='black', linewidth=2, linestyle='--')  # Thicker grid line at Y=132


cmap = plt.cm.get_cmap("Set1", len(np.unique(labels_initial)))
# Plot Initial Clusters
set_custom_grid(axs[0])
add_rectangles(axs[0], xyz_initial, labels_initial, cmap)
axs[0].set_xlabel('X [mm]', fontname='Arial', fontsize=18, fontweight='bold')
axs[0].set_ylabel('Y [mm]', fontname='Arial', fontsize=18, fontweight='bold')


# Set a scaling factor to determine the length of the arrow (you can adjust this value)
scaling_factor = -10  # This controls how long the direction vector is on the plot

before_beta_start = np.array([166.64386438, 127.89057459, 128.6296687])
before_beta_end = np.array([182.48591146, 150.27524609, 132.5197332])

after_beta_start = np.array([160.32913405, 127.98218572, 128.06892632])
after_beta_end = np.array([176.87258987, 137.94397229, 130.64358903])


# Before Beta
dx_before = before_beta_end[0] - before_beta_start[0]
dy_before = before_beta_end[1] - before_beta_start[1]

axs[0].arrow(
    before_beta_start[0], before_beta_start[1],
    dx_before, dy_before,
    head_width=2.5, head_length=3,
    fc='blue', ec='blue',
    linewidth=2,
    label='Before Beta',
    length_includes_head=True
)
axs[0].scatter(before_beta_start[0], before_beta_start[1], marker='x', color='blue', s=300)

# After Beta
dx_after = after_beta_end[0] - after_beta_start[0]
dy_after = after_beta_end[1] - after_beta_start[1]

axs[0].arrow(
    after_beta_start[0], after_beta_start[1],
    dx_after, dy_after,
    head_width=2.5, head_length=3,
    fc='green', ec='green',
    linewidth=2,
    label='After Beta',
    length_includes_head=True
)
axs[0].scatter(after_beta_start[0], after_beta_start[1], marker='x', color='green', s=300)

 

mean_vector_1 = np.array([159.31, 127.97, 127.99])

direction_vector_1 = np.array([0.85, 0.49, -0.2])

# Set a scaling factor to determine the length of the arrow (you can adjust this value)
scaling_factor = 29.56  # This controls how long the direction vector is on the plot

# Plot the mean point (starting point) and direction vector using ax.arrow
axs[0].arrow(mean_vector_1[0], mean_vector_1[1], direction_vector_1[0] * scaling_factor, direction_vector_1[1] * scaling_factor, 
          head_width=2, head_length=5, fc='k', ec='k')

axs[0].plot(mean_vector_1[0], mean_vector_1[1],
            marker='s', color='k', markersize=15,
            markeredgecolor='k', markeredgewidth=1.5,
            label='Mean Vector')


axs[0].text(0.05, 0.95, '(a)', transform=axs[0].transAxes, fontsize=16, fontweight='bold', verticalalignment='top')

beta_clusters = np.load('gmm_sigma_vs_beta_all_angles_10MeV.npy')

unique_cm_angles = np.unique(beta_clusters[:, 0])

# Use a colormap to assign distinct colors
colors = cm.viridis(np.linspace(0, 1, len(unique_cm_angles)))

for i, angle in enumerate(unique_cm_angles):
    rows = beta_clusters[beta_clusters[:, 0] == angle]

    x = rows[:, 1]
    y = rows[:, 2]

    axs[1].plot(
        x, y,
        color=colors[i],
        linestyle='-',
        marker='o',
        markersize=3,
        linewidth=3,
        label=fr"$\theta_{{\rm cm}} = {int(angle)}^\circ$"
    )


# Labels and legend
axs[1].set_xlabel(r'$\boldsymbol{\beta} * 100$', fontsize=18, fontweight='bold')
axs[1].set_ylabel(r'$\sigma(\theta_{labs} - \theta_{labr})^{\circ}$', fontsize=18, fontweight='bold')
axs[1].legend(fontsize=16)
axs[1].grid(True, linestyle="--", linewidth=0.5)

axs[1].text(0.05, 0.95, '(b)', transform=axs[1].transAxes, fontsize=16, fontweight='bold', verticalalignment='top')
axs[1].set_xlim([10,100])

plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.savefig('beta.png', dpi=300, bbox_inches='tight')
# 
fig.savefig("figure_beta.eps", format='eps', dpi=300)
plt.show()

C:\Users\alarokia\AppData\Local\Temp\ipykernel_9132\2259167050.py:58: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("Set1", len(np.unique(labels_initial)))
C:\Users\alarokia\AppData\Local\Temp\ipykernel_9132\2259167050.py:160: UserWarning: The figure layout has changed to tight
  plt.tight_layout(rect=[0, 0, 1, 0.95])
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
Traceback (most recent call last):
  File "C:\Users\alarokia\AppData\Local\Programs\Python\Python313\Lib\site-packages\matplotlib\backends\backend_qt.py", line 523, in _draw_idle
    self.draw()
    ~~~~~~~~~^^
  File "C:\Users\alarokia\AppData\Local\Programs\Python\Python313\Lib\site-packages\pylustrator\components\matplotlibwidget.py", line 78, in draw
    self.time